# LoRA: hands-on (optional)
**LLMs From Basics · Lesson M3.7**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

You'll write a LoRA layer from scratch in PyTorch, count its parameters, use it to teach a small **frozen** network a new task, and then merge the adapter back into the weights. Everything runs on a CPU in well under a minute (PyTorch is pre-installed on Colab).

Stuck? Solutions are at the very bottom.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
torch.set_num_threads(1)   # tiny models run fastest on one thread
print("torch", torch.__version__)

## 1 · A LoRA linear layer
Wrap an existing `nn.Linear` (weight $W_0$ of shape `out × in`) so that

$$h = W_0 x + b + \frac{\alpha}{r}\, B A x$$

- freeze the wrapped layer's weight and bias (`requires_grad_(False)`);
- `A` has shape `(r, in_features)`, initialised with small random values (here: `nn.init.kaiming_uniform_(A, a=math.sqrt(5))`, PEFT's default);
- `B` has shape `(out_features, r)`, initialised to **zeros**;
- `self.scaling = alpha / r`.

Inputs arrive as rows (`x` has shape `batch × in`), so the bypass is `x @ A.T @ B.T`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        self.r, self.alpha = r, alpha
        # TODO: freeze base, create self.A and self.B as nn.Parameter, set self.scaling
        raise NotImplementedError("Exercise 1")

    def forward(self, x):
        # TODO: base output plus the scaled low-rank bypass
        raise NotImplementedError("Exercise 1")

**Check 1.** At initialisation the adapter must be an exact no-op, and only `A` and `B` may train.

In [ ]:
lin = nn.Linear(32, 16)
lora = LoRALinear(lin, r=4, alpha=8)
x = torch.randn(5, 32)
assert torch.equal(lora(x), lin(x)), "B = 0, so the output must equal the base layer's"
assert not lin.weight.requires_grad and not lin.bias.requires_grad
assert lora.A.shape == (4, 32) and lora.B.shape == (16, 4)
assert lora.A.requires_grad and lora.B.requires_grad
assert lora.scaling == 2.0
print("✓ exercise 1")

## 2 · Count the parameters
Write `count_params(model, trainable_only)` and check the lesson's worked example: a 4096 × 4096 layer (no bias) with $r = 8$.

In [ ]:
def count_params(model, trainable_only=False):
    # TODO: sum p.numel() over model.parameters(), optionally only those with requires_grad
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
big = LoRALinear(nn.Linear(4096, 4096, bias=False), r=8)
total, trainable = count_params(big), count_params(big, trainable_only=True)
print(f"total {total:,} · trainable {trainable:,} · {100 * trainable / (4096 * 4096):.2f}% of W")
assert trainable == 65_536
assert total == 4096 * 4096 + 65_536
del big
print("✓ exercise 2")

## 3 · Teach a frozen network a new task
First we "pretrain" a small MLP on **task A**: classify 16-dimensional points by the sign of $w_A \cdot x$. Then the task changes to **task B**: the sign of $w_B \cdot x$, where $w_B$ points 60° away from $w_A$. A related task, but the pretrained model only gets about two thirds of task B right.

Your job: write `add_lora(model, r, alpha)`, which replaces **every** `nn.Linear` inside an `nn.Sequential` with a `LoRALinear` wrapping it, and freezes everything else. Then the provided loop fine-tunes only the adapters. (In a network this tiny the adapters are about 20% of the size of the base; with layer widths in the thousands, as in an LLM, it drops below 1%.)

In [ ]:
D_IN = 16
g = torch.Generator().manual_seed(1)
w_a = torch.randn(D_IN, generator=g)
u = torch.randn(D_IN, generator=g)
u = u - (u @ w_a) / (w_a @ w_a) * w_a                     # make u orthogonal to w_a
w_b = math.cos(math.radians(60)) * w_a / w_a.norm() + math.sin(math.radians(60)) * u / u.norm()

def make_data(w, n):
    X = torch.randn(n, D_IN, generator=g)
    return X, (X @ w > 0).long()

def make_mlp():
    return nn.Sequential(nn.Linear(D_IN, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(-1) == y).float().mean().item()

def fit(model, X, y, steps=200, lr=1e-2):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for _ in range(steps):
        idx = torch.randint(0, len(X), (128,), generator=g)
        loss = F.cross_entropy(model(X[idx]), y[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

XA, yA = make_data(w_a, 4000)
XB, yB = make_data(w_b, 2000)
XB_test, yB_test = make_data(w_b, 1000)
XA_test, yA_test = make_data(w_a, 1000)

torch.manual_seed(0)
pretrained = fit(make_mlp(), XA, yA)
print(f"pretrained: task A {accuracy(pretrained, XA_test, yA_test):.3f} · task B {accuracy(pretrained, XB_test, yB_test):.3f}")

In [ ]:
import copy

def add_lora(model, r=4, alpha=8):
    model = copy.deepcopy(model)          # leave the original untouched
    # TODO: freeze every parameter, then replace each nn.Linear at model[i] with LoRALinear(model[i], r, alpha)
    raise NotImplementedError("Exercise 3")
    return model

**Check 3.** Fine-tune only the adapters on task B.

In [ ]:
torch.manual_seed(0)
adapted = add_lora(pretrained, r=4, alpha=8)
n_train = count_params(adapted, trainable_only=True)
n_all = count_params(pretrained)
print(f"trainable {n_train:,} of {n_all:,} base parameters ({100 * n_train / n_all:.1f}%)")
assert torch.allclose(adapted(XB_test), pretrained(XB_test)), "a fresh adapter must not change the outputs"
assert n_train == 4 * (16 + 64) + 4 * (64 + 64) + 4 * (64 + 2)

before = {k: v.clone() for k, v in pretrained.state_dict().items()}
fit(adapted, XB, yB, steps=600)
acc_b = accuracy(adapted, XB_test, yB_test)
print(f"after LoRA fine-tuning: task B {acc_b:.3f}")
assert acc_b > 0.9
for name, p in adapted.named_parameters():
    if not (name.endswith(".A") or name.endswith(".B")):
        assert not p.requires_grad, name
assert all(torch.equal(before[k], v) for k, v in pretrained.state_dict().items()), "the base model must not change"
print("✓ exercise 3")

## 4 · Merge the adapter
For deployment, fold each adapter into its layer: $W' = W_0 + \frac{\alpha}{r} B A$, keeping the bias. Write `merge(model)` that returns a plain `nn.Sequential` with ordinary `nn.Linear` layers (no LoRA left), and check it gives the same outputs.

In [ ]:
def merge(model):
    layers = []
    for layer in model:
        if isinstance(layer, LoRALinear):
            base = layer.base
            # TODO: new nn.Linear with weight W0 + scaling * B @ A and the same bias
            raise NotImplementedError("Exercise 4")
        else:
            layers.append(copy.deepcopy(layer))
    return nn.Sequential(*layers)

**Check 4**

In [ ]:
merged = merge(adapted)
assert not any(isinstance(m, LoRALinear) for m in merged.modules())
assert count_params(merged) == n_all, "same size as the original model"
with torch.no_grad():
    out_m, out_a = merged(XB_test), adapted(XB_test)
print(f"max output difference after merging: {(out_m - out_a).abs().max().item():.2e}  (float32 rounding)")
assert torch.allclose(out_m, out_a, rtol=1e-4, atol=1e-4)
assert abs(accuracy(merged, XB_test, yB_test) - acc_b) < 0.002
print("✓ exercise 4")

## 5 · How much rank do you need?
Fine-tune fresh adapters with $r \in \{1, 2, 4, 8\}$ (keep $\alpha = 2r$) and record task-B test accuracy in the dict `rank_acc`. Then look at the plot: what does task B *need*, given that it is "rotate one direction into another"?

In [ ]:
rank_acc = {}
for r in [1, 2, 4, 8]:
    torch.manual_seed(0)
    # TODO: add_lora(pretrained, r, 2 * r), fit on task B (600 steps), store test accuracy
    pass

**Check 5**

In [ ]:
print(rank_acc)
assert set(rank_acc) == {1, 2, 4, 8}
assert min(rank_acc.values()) > 0.85, "even tiny ranks should learn this task"
plt.figure(figsize=(5, 3))
plt.plot(list(rank_acc), list(rank_acc.values()), "o-")
plt.xscale("log", base=2); plt.xlabel("rank r"); plt.ylabel("task B accuracy"); plt.ylim(0.5, 1.0)
plt.show()
print("✓ exercise 5")

## Experiment
- What does the adapted model now score on **task A**? Compare with the pretrained model. (Fine-tuning can cause *forgetting*, even with LoRA.) Then compute task-A accuracy for the base model and the adapted one side by side.
- Put LoRA on the **last layer only**. Is that enough here? Why might the first layer matter for this task?
- Try a 10× smaller learning rate, or $\alpha = r$ instead of $2r$. Which knob does $\alpha$ really behave like?

In [ ]:
print(f"task A · pretrained {accuracy(pretrained, XA_test, yA_test):.3f} · adapted {accuracy(adapted, XA_test, yA_test):.3f}")

## Recap
- LoRA freezes $W_0$ and learns $\frac{\alpha}{r}BA$; with $B = 0$ at the start, the adapted model **begins exactly** at the pretrained one.
- Trainable parameters per layer: $r(d_\text{in} + d_\text{out})$, a tiny fraction of $d_\text{in} d_\text{out}$.
- After training, **merge** with $W' = W_0 + \frac{\alpha}{r}BA$: same shape, same outputs, no extra cost at inference.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        self.r, self.alpha = r, alpha
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
import copy

def add_lora(model, r=4, alpha=8):
    model = copy.deepcopy(model)
    for p in model.parameters():
        p.requires_grad_(False)
    for i, layer in enumerate(model):
        if isinstance(layer, nn.Linear):
            model[i] = LoRALinear(layer, r, alpha)
    return model

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def merge(model):
    layers = []
    for layer in model:
        if isinstance(layer, LoRALinear):
            base = layer.base
            new = nn.Linear(base.in_features, base.out_features, bias=base.bias is not None)
            with torch.no_grad():
                new.weight.copy_(base.weight + layer.scaling * layer.B @ layer.A)
                if base.bias is not None:
                    new.bias.copy_(base.bias)
            layers.append(new)
        else:
            layers.append(copy.deepcopy(layer))
    return nn.Sequential(*layers)

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
rank_acc = {}
for r in [1, 2, 4, 8]:
    torch.manual_seed(0)
    m = fit(add_lora(pretrained, r, 2 * r), XB, yB, steps=600)
    rank_acc[r] = accuracy(m, XB_test, yB_test)